### Load Healthcare Data From Excel File To Populate Bronze Level Tables (PostgreSQL)

- bronze_patient
- bronze_patient_encounter


In [5]:
# import libraries
import pandas as pd
# import libraries from psycog2 and sqlalchemy to connect to PostgreSQL 18
import psycopg2
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os


In [18]:
# load .env; loads credentials into memeory 
load_dotenv(override=True)

True

In [19]:
host=os.getenv('DB_HOST')
user=os.getenv('DB_USER')
password=os.getenv('DB_PASSWORD')
database=os.getenv('DB_NAME')
port=os.getenv('DB_PORT')

In [20]:
print(host, user,password, database, port)

localhost postgres simien24 healthcare 5432


In [8]:
# load data from Excel sheets
dict_df = pd.read_excel('diabetic_data_load.xlsx', sheet_name=[0,1])

In [9]:
# create patient and patient encounter data frames
patient_df = dict_df[0]
patient_encounter_df = dict_df[1]

In [10]:
# Data check/validate
patient_df.head()

,patient_nbr,race,gender,age,weight
0,8222157,Caucasian,Female,[0-10),?
1,55629189,Caucasian,Female,[10-20),?
2,86047875,AfricanAmerican,Female,[20-30),?
3,82442376,Caucasian,Male,[30-40),?
4,42519267,Caucasian,Male,[40-50),?


In [11]:
patient_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   patient_nbr  101766 non-null  int64
 1   race         101766 non-null  str  
 2   gender       101766 non-null  str  
 3   age          101766 non-null  str  
 4   weight       101766 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.9 MB


In [12]:
# Data check/validate
patient_encounter_df.head()

,encounter_id,patient_nbr,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,...,citoglipton,insulin,glyburideMetformin,glipizideMetformin,glimepiridePioglitazone,metforminRosiglitazone,metforminPioglitazone,change_in_meds,diabetesMed,readmitted
0,2278392,8222157,6,25,1,1,?,Pediatrics-Endocrinology,41,0,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,1,1,7,3,?,?,59,0,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,1,1,7,2,?,?,11,5,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,1,1,7,2,?,?,44,1,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,1,1,7,1,?,?,51,0,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [13]:
patient_encounter_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 46 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   admission_type_id         101766 non-null  int64 
 3   discharge_disposition_id  101766 non-null  int64 
 4   admission_source_id       101766 non-null  int64 
 5   time_in_hospital          101766 non-null  int64 
 6   payer_code                101766 non-null  str   
 7   medical_specialty         101766 non-null  str   
 8   num_lab_procedures        101766 non-null  int64 
 9   num_procedures            101766 non-null  int64 
 10  num_medications           101766 non-null  int64 
 11  number_outpatient         101766 non-null  int64 
 12  number_emergency          101766 non-null  int64 
 13  number_inpatient          101766 non-null  int64 
 14  diag_1         

In [21]:
# create connection to PostgreSQL db: healthcare
connection_string = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}"
engine = create_engine(connection_string)

# load dataframe into bronze_patient
table_name = 'bronze_patient'
patient_df.to_sql(table_name, engine, if_exists='replace', index=False)

print(f"Data successfully loaded into table '{table_name}' in database '{database}'.")



Data successfully loaded into table 'bronze_patient' in database 'healthcare'.


In [22]:
# perform data check of bronze_patient
connection_string = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}"
try:
    engine = create_engine(connection_string)

    with engine.connect() as conn:
        bronze_patient = pd.read_sql('''SELECT * FROM bronze_patient;''', con = conn)

        conn.close()

except (Exception) as error:
        error_log   = 'Error writing data to table: bronze_patient. Check system file.'                  
        with open('error_log.txt','w') as error:
           error.write(error_log)


bronze_patient.head()

,patient_nbr,race,gender,age,weight
0,8222157,Caucasian,Female,[0-10),?
1,55629189,Caucasian,Female,[10-20),?
2,86047875,AfricanAmerican,Female,[20-30),?
3,82442376,Caucasian,Male,[30-40),?
4,42519267,Caucasian,Male,[40-50),?


In [23]:
bronze_patient.info()

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   patient_nbr  101766 non-null  int64
 1   race         101766 non-null  str  
 2   gender       101766 non-null  str  
 3   age          101766 non-null  str  
 4   weight       101766 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.9 MB


In [24]:
# create connection to PostgreSQL db: healthcare
connection_string = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}"
engine = create_engine(connection_string)

# load dataframe into bronze_patient_encounter
table_name = 'bronze_patient_encounter'
patient_encounter_df.to_sql(table_name, engine, if_exists='replace', index=False)

print(f"Data successfully loaded into table '{table_name}' in database '{database}'.")


Data successfully loaded into table 'bronze_patient_encounter' in database 'healthcare'.


In [25]:
# Data Check: extract data from bronze_patient_encounter table
connection_string = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}"
try:
    engine = create_engine(connection_string)

    with engine.connect() as conn:
        bronze_patient_encounter = pd.read_sql('''SELECT * FROM bronze_patient_encounter;''', con = conn)

        conn.close()

except (Exception) as error:
        error_log   = 'Error writing data to table: bronze_patient_encounter. Check system file.'                  
        with open('error_log.txt','w') as error:
           error.write(error_log)


bronze_patient_encounter.head()

,encounter_id,patient_nbr,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,...,citoglipton,insulin,glyburideMetformin,glipizideMetformin,glimepiridePioglitazone,metforminRosiglitazone,metforminPioglitazone,change_in_meds,diabetesMed,readmitted
0,2278392,8222157,6,25,1,1,?,Pediatrics-Endocrinology,41,0,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,1,1,7,3,?,?,59,0,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,1,1,7,2,?,?,11,5,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,1,1,7,2,?,?,44,1,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,1,1,7,1,?,?,51,0,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [26]:
bronze_patient_encounter.info()

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 46 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   encounter_id              101766 non-null  int64
 1   patient_nbr               101766 non-null  int64
 2   admission_type_id         101766 non-null  int64
 3   discharge_disposition_id  101766 non-null  int64
 4   admission_source_id       101766 non-null  int64
 5   time_in_hospital          101766 non-null  int64
 6   payer_code                101766 non-null  str  
 7   medical_specialty         101766 non-null  str  
 8   num_lab_procedures        101766 non-null  int64
 9   num_procedures            101766 non-null  int64
 10  num_medications           101766 non-null  int64
 11  number_outpatient         101766 non-null  int64
 12  number_emergency          101766 non-null  int64
 13  number_inpatient          101766 non-null  int64
 14  diag_1                    10176